# **Batch Deployment of ML Models**

In this notebook we will prepare our code for **batch deployment**.  
We will use the **MLFlow server** we set up in the previous notebook to **store our models** and the **predictions**. Afterwards, we will **refactor our code** into **Python scripts** and **schedule** the execution of the predictions with **Prefect**.

**Batch predictions** are predictions that are executed on a **regular basis**. For example, you might want to **predict the sales** for the next day **every night at 12am**.  
Batch predictions are often used in **analytics** when there is **no need for real-time predictions**. 

In this example, we want to **analyse** if there are differences between the **actual duration of a trip** and the **duration predicted by the model**.  
It is not the **best example** for batch predictions, but it is **good enough** to show how it works.

First let's **import the libraries** we will need:

In [ ]:
import os
import uuid
import pandas as pd

import mlflow

from dotenv import load_dotenv


Before you run the next cell, it is important to add to your `.env` file the following variables:
+ `RUN_ID` of the model you want to use for predictions. You can find the `RUN_ID` in the MLFlow UI, in the "Experiments" section. It is a string of 32 characters. Make sure to use the one of the model you trained in the previous notebook with the pipeline.
+ `BUCKET_NAME` of your GCP bucket where the MLFLOW artifacts get stored. It is the same bucket you used to set up the MLFlow server. For example `mlflow-artifacts/models` (see [here](./01-setup-mlflow-server.md#create-a-gcs-bucket)). 



In [ ]:
# Load environment variables from the .env file
load_dotenv()

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
RUN_ID = os.getenv("RUN_ID")
BUCKET_NAME = os.getenv("BUCKET_NAME")
GOOGLE_APPLICATION_CREDENTIALS = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Set the MLflow tracking URI
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [ ]:
# Test that the credentials for Google cloud are working by listing the buckets
from google.cloud import storage
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(
    GOOGLE_APPLICATION_CREDENTIALS
)

# This will automatically use the credentials from the environment variable
storage_client = storage.Client(credentials=credentials)

# Example: List the buckets in your project
buckets = list(storage_client.list_buckets())
print("Buckets:")
for bucket in buckets:
    print("-----")
    print(f" - {bucket}")
    print(f" - {bucket.name}")
    print(f" - {bucket.location}")
    print(f" - {bucket.time_created}")
    print("-----")

Now we will download the dataset needed for batch predictions. In the previous notebook, we trained the model using the Green Taxi data from July 2025. For this notebook, we will use the August 2025 Green Taxi data to generate batch predictions.

In [ ]:
# Download the dataset using pandas
year = 2025
month = 8
color = "green"

parquet_file = f"{color}_tripdata_{year}-{month:02d}.parquet"
file_url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/" + parquet_file

df_download = pd.read_parquet(file_url)

save_path = "data"
file_path_input = f"{save_path}/{parquet_file}"

if not os.path.exists(file_path_input):
    df_download.to_parquet(f"{save_path}/{parquet_file}", index=False)


## Prepare the code for batch deployment

Normally data comes with **unique identifiers**. In our case we don't have any **unique identifiers** so we will create one for each row. We will use the `uuid` library to them.

In [ ]:
def generate_uuids(n):
    ride_ids = []
    for i in range(n):
        ride_ids.append(str(uuid.uuid4()))
    return ride_ids


Let's write a function that loads the data, creates the trip_duration column and adds the unique identifier:

In [ ]:
def read_dataframe(file_path_input: str):
    df = pd.read_parquet(file_path_input)

    df['trip_duration_minutes'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.trip_duration_minutes = df.trip_duration_minutes.dt.total_seconds() / 60
    df = df[(df.trip_duration_minutes >= 1) & (df.trip_duration_minutes <= 60)]
    
    df['ride_id'] = generate_uuids(len(df))

    return df

We need to prepare the data the same way as in the notebook before. We will use the same function to do that:

In [ ]:
def preprocess(df):
    df = df.copy()
    categorical_features = ["PULocationID", "DOLocationID"]
    df[categorical_features] = df[categorical_features].astype(str)
    
    df['trip_route'] = df["PULocationID"] + "_" + df["DOLocationID"]
    dicts = df[['trip_route', 'trip_distance']].to_dict(orient='records')
    
    return dicts

Now we can write the functions that load the model from MLFlow, makes the predictions and creates a new dataframe with the predictions. We will use  the following functions:
1. `load_model`: loads the model from MLFlow usisng the `RUN_ID` and the model name (default is `mlflow-model-v1`) that we used to log the model in the previous notebook. In case you have given a different name to the model, you can change it here.
2. `save_results`: saves the predictions to a parquet file.
3. `apply_model`: read the data, preprocess it, load the model, make the predictions and save the results.

In [ ]:
def load_model(mlflow_tracking_uri, run_id, model_name='mlflow-model-v1', ):
    mlflow.set_tracking_uri(mlflow_tracking_uri)
    logged_model_name = f'runs:/{run_id}/{model_name}'
    # Load model as a PyFuncModel.
    loaded_model = mlflow.pyfunc.load_model(logged_model_name)
    return loaded_model


def save_results(df, y_pred, run_id, model_name, file_path_predictions):
    df_result = pd.DataFrame()
    df_result['ride_id'] = df['ride_id']
    df_result['lpep_pickup_datetime'] = df['lpep_pickup_datetime']
    df_result['PULocationID'] = df['PULocationID']
    df_result['DOLocationID'] = df['DOLocationID']
    df_result['actual_duration'] = df['trip_duration_minutes']
    df_result['predicted_duration'] = y_pred
    df_result['diff'] = df_result['actual_duration'] - df_result['predicted_duration']
    df_result['model_version'] = run_id
    df_result['model_name'] = model_name
    df_result.to_parquet(file_path_predictions, index=False)



def apply_model(file_path_input, mlflow_tracking_uri, run_id, model_name, file_path_predictions):
    df = read_dataframe(file_path_input)
    dicts = preprocess(df)
    
    loaded_model = load_model(mlflow_tracking_uri, run_id, model_name)
    y_pred = loaded_model.predict(dicts)
    
    save_results(df, y_pred, run_id, model_name, file_path_predictions)


In [ ]:
# Now we can apply the model to the new data
# and save the predictions to a new parquet file
apply_model(file_path_input=file_path_input, 
mlflow_tracking_uri=MLFLOW_TRACKING_URI,
run_id=RUN_ID, model_name="mlflow-model-v1",
file_path_predictions="data/predictions.parquet")

In [ ]:
# Let's read the predictions
pd.read_parquet("data/predictions.parquet")

## **Refactor the code**

Now we can refactor this code into python scripts. In the folder called `src/batch/` we will create a file named `predict.py`. We will copy the code from the functions above into the scripts. What we will add in the end is a `def run()` function that will run the whole script. The `run()` function will be called in the `if __name__ == "__main__":` block. And will be parameterized with click:

```python
@click.command()
@click.option("--file_path_input", help="Path to the input parquet file")
@click.option("--mlflow_tracking_uri", help="MLflow tracking URI")
@click.option("--run_id", help="MLflow run ID")
@click.option("--model_name", help="MLflow model name")
@click.option("--file_path_predictions", help="Path to the output parquet file")
@click.option("--google_sa_key", help="Path to the Google Service Account Key JSON file")
def run(file_path_input, mlflow_tracking_uri, run_id, model_name, file_path_predictions, google_sa_key):
    file_path_input = file_path_input
    file_path_predictions = file_path_predictions
    run_id = run_id
    model_name = model_name
    # Set the environment variable for Google Application Credentials
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = google_sa_key
    apply_model(file_path_input,
                mlflow_tracking_uri,
                run_id,
                model_name,
                file_path_predictions)
    
    

if __name__ == "__main__":
    run()
```

Open the Terminal and move to the folder where the `src` folder is located. Now you can run the `predict.py` script with the following command:

```bash
python src/batch/predict.py --help
```

i.e. you can try to run the file as below substituting with the correct parameters.  

```zsh
python src/batch/predict.py \
    --file_path_input 'data/green_tripdata_2025-08.parquet' \
    --mlflow_tracking_uri '<your-mlflow-tracking-uri>' \
    --run_id '<your-mlflow-run-id>' \
    --model_name '<your-mlflow-model-name>' \
    --file_path_predictions 'data/predictions_batch.parquet' \
    --google_sa_key '<path-to-your-service-account.json>'
```

After you run you will find a new prediction file in the data folder called predictions_batch.parquet. What you have now is a command that can be run anytime you want to create predictions. You could schedule this command to be automatically run regularly - i.e. once a day, once a week or once a month. 

Also feel free to change the filename into a command that downloads also other data from the green taxi dataset. In the end you could also create a Docker image from this script and run it in a container.

| **Argument**              | **Description**                                                                                                         |
| ------------------------- | ----------------------------------------------------------------------------------------------------------------------- |
| `--file_path_input`       | Path to the input data file in Parquet format that will be used for generating predictions.                             |
| `--mlflow_tracking_uri`   | The tracking server URI used by MLflow to locate runs, models, and artifacts.        |
| `--run_id`                | The MLflow run ID corresponding to the specific model version you want to use for inference.                            |
| `--model_name`            | The registered name of the MLflow model you want to load for predictions.                                               |
| `--file_path_predictions` | Path where the output predictions file will be saved in Parquet format.                                                 |
| `--google_sa_key`         | Path to the Google Cloud service account JSON key, used for authentication when accessing remote MLflow or GCS storage. |


## **Scheduling Predictions with Prefect**

In this section, we will **schedule the batch predictions** using **Prefect** — a modern **workflow orchestration system** that makes it easy to **automate, schedule, and monitor** your data pipelines.

Prefect allows you to define your Python workflows (called *flows*) and then execute them automatically on a schedule, either **locally** or using **Prefect Cloud**.

You can learn more from the official [Prefect documentation](https://docs.prefect.io/v3/) and review the **Workflow Orchestration** materials from the previous week for background.

### **Step 1 — Start a Prefect Server (Locally)**

In this example, we’ll run the **Prefect Server locally**, which provides:
- A **REST API** for scheduling and managing flows  
- A **web UI** for monitoring flow runs  

Open a **new terminal window** and start the Prefect server:

```bash
prefect server start
```

You should see logs similar to:
![prefect server start](./images/prefect_server_start.png)

---

### **Step 2 — Configure Prefect to Use the Local Server**
Once the Prefect server is running, Prefect needs to know **where** to send API requests.  
You’ll do this by setting the environment variable `PREFECT_API_URL` so that your local Prefect client connects to the running server.

Open **another terminal window** (keep the server running in the first one and make sure to be in the same Python environment where Prefect is installed), and run:

```bash
prefect config set PREFECT_API_URL=http://127.0.0.1:4200/api
```
This tells Prefect to use the server’s API, which by default runs at `http://127.0.0.1:4200/api`.

To make sure the configuration was set correctly, you can run in the same terminal:

#### Test the configuration
```bash
prefect config view
```

You should see:
```bash
PREFECT_PROFILE='local'
PREFECT_API_URL='http://127.0.0.1:4200/api'
```
#### Test the connection to the API

You can also check that the Prefect API is live by visiting:
[http://127.0.0.1:4200/docs](http://127.0.0.1:4200/docs).
This opens the Prefect interactive API documentation (Swagger UI), which confirms your local server is running and accessible. Please feel free to explore the available endpoints.

---

### **Step 3 — View the Prefect Dashboard and Stop the Server**

Once you see that the server is running, you can open the **Prefect UI** in your browser at: [http://127.0.0.1:4200/dashboard](http://127.0.0.1:4200/dashboard).

You should see the Prefect dashboard interface:

![Prefect Dashboard](./images/prefect_dashboard.png)

From this dashboard, you’ll be able to:
- View your registered **flows**, and **deployments** ...
- Monitor active and scheduled **runs**
- Inspect **logs**, **schedules**, and **results**



#### Stopping the Server

When you’re finished working, you canstop the server you can use `CTRL+C` in the terminal window where the server is running.

---


### **Step 4 — Refactor the Code into a Prefect Flow**
In the previous section, we used an **input file** that we **downloaded locally with Pandas**, saved to disk, and then used to **generate predictions**.  

Now, we’ll take that same process and make it **cloud-native** and **orchestrated with Prefect**.  
Instead of using local files, we’ll work directly with a **Google Cloud Storage (GCS) bucket**, which we’ve already created.  
This bucket will store both the **input data** and the **output predictions**, organized in dedicated folders.

With **Pandas**, we can **read and write Parquet files** directly **from and to GCS**, making the process fast, efficient, and easy to automate.


#### 4.1 Create a Flow file

We will create a new Python file called `flow_prefect.py` and **copy the code** from the existing `predict.py` script into it.
We’ll make a few key changes to transform it into a Prefect-managed workflow:
1. **Remove Click options** — Prefect will handle parameters through deployments or schedules.  
2. **Add parameters** as normal Python function arguments.  
3. **Import Prefect’s decorators** and utilities (`flow`, `task`, `get_run_logger`, and `get_run_context`).  
4. **Convert** the main `run()` function into a Prefect **flow** using `@flow`.  
5. **Wrap** the model inference step (`apply_model`) in a Prefect **task** using `@task`.

at the top of the new file `flow_prefect.py`, we will remove the click import and add the Prefect imports and other necessary imports:

```python
...
from datetime import date
from prefect import task, flow, get_run_logger
from prefect.context import get_run_context
```

#### 4.2 Define the Flow Functions
We’ll turn the `run()` function into a Prefect flow by decorating it with `@flow`. This allows Prefect to orchestrate, schedule, and monitor each run.

```python
@flow(name="Predict Green Taxi Trip Duration")
def run(
    bucket_name: str,
    mlflow_tracking_uri: str,
    run_id: str,
    model_name: str,
    google_sa_key: str,
    data_reference_date: date
):
```
<div style="
  display:block;
  max-width:90%;
  padding:0.75rem 1rem;
  border-left:4px solid #0957de;
  background:#f5f9ff;
  color:#093170;
  border-radius:4px;
  font-family:system-ui, Segoe UI, Roboto, sans-serif;
  font-size:0.95rem;
  line-height:1.5;
  text-indent:-2.2rem;
  padding-left:3rem;">
  💡 <strong style="color:#0957de;">Note:</strong> We added a new parameter <code>data_reference_date</code> to the <code>run()</code> function. This allows us to specify a custom reference date for determining which month of data to process.
</div>

#### 4.3 Add Logging
To track the progress of our flow and see detailed logs in the Prefect UI, we use Prefect’s built-in `logger`

```python
    logger = get_run_logger()
    logger.info(
        f"Running with parameters: bucket_name={bucket_name}, run_id={run_id}, google_sa_key={google_sa_key}"
    )
    path = os.path.dirname(__file__)
    logger.info(f"Current path: {path}")

    ...
    logger.info(f"Processing data for {year}-{month:02d}")

    ...
    logger.info(f"Raw data saved to {raw_path}")

    ...
    logger.info(f"Raw data saved to {raw_path}")
```

#### 4.4 Determine the Reference Date
Our flow always processes the **previous month’s data**. We compute the correct period based on a provided date.

```python
    # Use the provided data_reference_date 
    reference_date = data_reference_date

    # Compute the previous month based on the reference date
    year = reference_date.year
    month = reference_date.month - 1
    if month == 0:
        month = 12
        year -= 1
```

#### 4.5 Connect to GCS, Download the Data from the Public URL, Upload it to GCS, and Run Predictions 

We use the `Google Service Account` credentials to authenticat and handle data directly in GCS. Furthermore, we download the data for the previous month from the public NYC TLC URL, upload it to our GCS bucket, and then run the predictions with the model we trained and stored in MLFlow:
```python
    # Set the environment variable for Google Application Credentials
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = google_sa_key
    
    # Read the data for the previous month from the public URL
    df_input = pd.read_parquet(f"https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_{year}-{month:02d}.parquet")
    logger.info(f"Read {len(df_input)} rows.")
    
    # Save raw data to the bucket
    raw_path = f"gs://{bucket_name}/raw/green_tripdata_{year}-{month:02d}.parquet"
    df_input.to_parquet(raw_path)
    logger.info(f"Raw data saved to {raw_path}")

    
    # Define input and output filenames in GCS
    file_path_input = raw_path
    file_path_prediction = f"gs://{bucket_name}/predictions/green_tripdata_{year}-{month:02d}.parquet"
    
    mlflow_tracking_uri = mlflow_tracking_uri
    run_id = run_id
    model_name = model_name

    # Run model inference task
    apply_model(file_path_input,
                mlflow_tracking_uri,
                run_id,
                model_name,
                file_path_prediction,
                )
```

<div style="
  display:block;
  max-width:80%;
  padding:0.75rem 1rem;
  border-left:4px solid #b42318;
  background:#fff5f5;
  color:#7a271a;
  border-radius:4px;
  font-family:system-ui, -apple-system, Segoe UI, Roboto, sans-serif;
  font-size:0.95rem;
  line-height:1.5;
  text-indent:-2.4rem;
  padding-left:3rem;">
  ⚠️ <strong style="color:#b42318;">Warning:</strong> The dataset for the selected month might not be available yet on the NYC TLC server. If you encounter an <code>HTTP 403: Forbidden</code> error, try setting <code>DATA_REFERENCE_DATE</code> to an earlier month. 
  <br>See <strong>Step 5 — Add <code>Data_REFERENCE_DATE</code> to the .env File</strong>.
</div>

#### 4.6 Define the Model Inference as a Prefect Task
We’ll convert the `apply_model()` function into a Prefect **task** by decorating it with `@task`. This allows Prefect to manage retries, logging, and execution of this step. We will also add logging to track the progress of the task.

```python
@task(name="apply_model_task")
def apply_model(file_path_input: str, 
                mlflow_tracking_uri: str, 
                run_id: str, 
                model_name: str, 
                file_path_predictions: str):
    """
    Task to apply the MLflow model and save predictions to GCS.
    """
    logger = get_run_logger()
    logger.info(f"Reading data from {file_path_input}")
    df = read_dataframe(file_path_input)
    logger.info(f"Preprocessing {df} rows")
    
    dicts = preprocess(df)
    
    logger.info(f"Loading the model {run_id}")
    loaded_model = load_model(mlflow_tracking_uri, run_id, model_name)
    logger.info(f"Model flavor: {loaded_model.metadata.flavors}")
    logger.info(f"{type(loaded_model._model_impl.sklearn_model)}")

    y_pred = loaded_model.predict(dicts)
    
    logger.info(f"Saving results to {file_path_predictions}")
    save_results(df, y_pred, run_id, model_name, file_path_predictions)
```

---

### **Step 5 — Add `Data_REFERENCE_DATE` to the .env File**
Before creating the deployment, let’s add a new variable to the `.env` file:

```bash
DATA_REFERENCE_DATE=2025-09-01
```
This parameter defines the **reference date** that the flow uses to determine which month’s data to process.


### **Step 6 — Create a Prefect Deployment**

Now that our Prefect flow is defined in `flow_prefect.py`, it’s time to make it **deployable and schedulable**.  

In Prefect, a **deployment** is a specific configuration of a flow that includes:
- Its **schedule** (when it should run)
- Any **default parameters**
- Optional **tags** and **metadata**

Deployments let you run your flow automatically — either on a **regular schedule** or in response to **specific events**.

#### 5.1 Create a Deployment Script
We’ll create a new Python file called `prefect_deploy.py` in the same folder as `flow_prefect.py`.  
This script will:
1. Import your Prefect flow  
2. Load environment variables from `.env`  
3. Define a schedule using an [RFC 5545 RRule](https://datatracker.ietf.org/doc/html/rfc5545)  
4. Serve the flow deployment locally using `run.serve()` so Prefect can execute it automatically

Here’s the complete code:

```python
import os
from dotenv import load_dotenv
from datetime import datetime, timezone, timedelta
from flow_prefect import run
from prefect.client.schemas.schedules import RRuleSchedule, CronSchedule 


# Load environment variables from a .env file
load_dotenv()

# Start the first run 2 minutes from now, then every 2 months on the same day and time
now = datetime.now(timezone.utc) + timedelta(minutes=2)
rrule_string = (
    f"DTSTART:{now.strftime('%Y%m%dT%H%M%SZ')}\n"
    f"FREQ=MONTHLY;INTERVAL=2;BYMONTHDAY={now.day};BYHOUR={now.hour};BYMINUTE={now.minute}"
)


schedule_rule = RRuleSchedule(rrule=rrule_string)


# Start the flow with a schedule
run.serve(
    name="ride_duration_prediction",
    schedules=[
        schedule_rule,
        ],
    parameters={
        "bucket_name": os.getenv("BUCKET_NAME"),
        "mlflow_tracking_uri": os.getenv("MLFLOW_TRACKING_URI"),
        "run_id": os.getenv("RUN_ID"),
        "model_name": os.getenv("MODEL_NAME"),
        "google_sa_key": os.getenv("GOOGLE_APPLICATION_CREDENTIALS"),
        "data_reference_date":os.getenv("DATA_REFERENCE_DATE") if os.getenv("DATA_REFERENCE_DATE") else None,
    },
    tags=["batch", "predict", "prefect"],
)
```
**What this Schedule does**:
- Starts the first flow run **two minutes after you execute the script**,so you can test it immediately.
- Then repeats **every two months**, on the **same day and time** as that first run.
- You can modify the rule to any schedule you like.

For example, to run the flow **every month on the 2nd at 3:00am**, you could use the following `rrule_string`  in the script:

```python
# Run on the 2nd day of every month at 3:00 AM
rrule_string = "FREQ=MONTHLY;BYMONTHDAY=2;BYHOUR=3;BYMINUTE=0"
```


**What the `run.serve()` function does behind the scenes is**:

1. **Registers your flow** with the Prefect API or server.  
   - You’ll see it appear under **Deployments** in the Prefect UI.  
2. **Creates and applies the schedule** you defined (e.g., via `RRuleSchedule` or `CronSchedule`).  
3. **Starts a lightweight background process** that continuously polls the Prefect server for upcoming scheduled runs.  
4. **Passes the parameters** you specified to each flow run.

---

#### **5.2 — Run the Deployment**

Now you can run the deployment script to **register your flow** and **start the schedule**.

Open a **new terminal window** — keep your Prefect server running in the first one —  
and make sure you’ve **activated the same Python environment** where Prefect is installed.

Then, run one of the following commands depending on your current directory:

If you are in the **project root**:
```bash
python src/batch/prefect_deploy.py
```
If you are in the **src/batch** folder:
```bash
python prefect_deploy.py
```
Once executed , Prefect will:
1. Register your flow as a deployment with the Prefect server
2. Start serving and polling for scheduled runs
3. Display a confirmation message in your terminal
   

You should see something like this — and the terminal window will remain open, continuously polling for upcoming scheduled runs:
```bash
Your flow 'Predict Green Taxi Trip Duration' is being served and polling for scheduled runs!

To trigger a run for this flow, use the following command:

        $ prefect deployment run 'Predict Green Taxi Trip Duration/ride_duration_prediction'

You can also run your flow via the Prefect UI: http://127.0.0.1:4200/deployments/deployment/deployment_id_here
```

<div style="
  display:block;
  max-width:80%;
  padding:0.75rem 1rem;
  border-left:4px solid #0957de;
  background:#f5f9ff;
  color:#093170;
  border-radius:4px;
  font-family:system-ui, -apple-system, Segoe UI, Roboto, sans-serif;
  font-size:0.95rem;
  line-height:1.5;
  text-indent:-2.2rem;
  padding-left:3rem;">
  💡 <strong style="color:#0957de;">Note:</strong> The first run of the deployment is scheduled to start within about two minutes after you execute the script. 
  You can follow the link provided in the terminal output if you want to run the flow directly via the Prefect UI (see the first image below), then click the <strong>Run</strong> button. 
  Alternatively, you can visit 
  <a href="http://127.0.0.1:4200/flows" target="_blank" style="color:#0957de; text-decoration:none; font-weight:500;">http://127.0.0.1:4200/flows</a> (see the second image below). 
  From there, click on <strong>Next Run</strong> to view the scheduled execution and open the <strong>Logs</strong> tab to follow the flow’s progress in real time.
</div>




![](./images/prefect_manual_run_gui.png)

![Prefect Deployment Running](./images/prefect_next_run.png)